In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/processed/player_seasons_raw.csv")

df["FG3A_total"] = df["FG3A"] * df["GP"]
df["FTA_total"]  = df["FTA"] * df["GP"]

for t in [1, 10, 20, 30, 50]:
    print(t,
          round((df["FG3A_total"] < t).mean(), 3),
          round((df["FTA_total"] < t).mean(), 3))

1 0.11 0.051
10 0.232 0.182
20 0.299 0.263
30 0.342 0.329
50 0.412 0.439


In [3]:
df["season_year"] = df["SEASON"].str[:4].astype(int)   # "2022-23" -> 2022

In [4]:
lookup = df[["PLAYER_ID", "season_year", "PTS"]]

def add_shifted(df, k, name):
    s = lookup.assign(season_year=lookup["season_year"] + k)
    s = s.rename(columns={"PTS": name})
    return df.merge(s, on=["PLAYER_ID", "season_year"],
                    how="left", validate="one_to_one")

In [5]:
df = add_shifted(df, -1, "ppg_next")
for k in [1, 2, 3]:
    df = add_shifted(df, k, f"ppg_lag{k}")

df["target"] = df["ppg_next"] - df["PTS"]

In [6]:
print(df[["PTS", "ppg_lag1", "ppg_lag2", "ppg_lag3", "ppg_next", "target"]].isna().mean())
print(df["target"].isna().groupby(df["SEASON"]).mean())
print(df["target"].describe())

PTS         0.000000
ppg_lag1    0.271103
ppg_lag2    0.439499
ppg_lag3    0.569576
ppg_next    0.271103
target      0.271103
dtype: float64
SEASON
2013-14    0.180498
2014-15    0.195122
2015-16    0.189076
2016-17    0.193416
2017-18    0.237037
2018-19    0.245283
2019-20    0.177694
2020-21    0.166667
2021-22    0.284298
2022-23    0.165121
2023-24    0.218531
2024-25    0.184534
2025-26    1.000000
Name: target, dtype: float64
count    5060.000000
mean       -0.051838
std         3.300841
min       -19.900000
25%        -2.100000
50%        -0.100000
75%         1.900000
max        15.500000
Name: target, dtype: float64


In [7]:
cols = ["SEASON", "PTS", "ppg_lag1", "ppg_lag2", "ppg_lag3", "ppg_next", "target"]
print(df[df["PLAYER_NAME"] == "LeBron James"][cols])
print(df[df["PLAYER_NAME"] == "Klay Thompson"][cols])

       SEASON   PTS  ppg_lag1  ppg_lag2  ppg_lag3  ppg_next  target
287   2013-14  27.1       NaN       NaN       NaN      25.3    -1.8
783   2014-15  25.3      27.1       NaN       NaN      25.3     0.0
1265  2015-16  25.3      25.3      27.1       NaN      26.4     1.1
1742  2016-17  26.4      25.3      25.3      27.1      27.5     1.1
2261  2017-18  27.5      26.4      25.3      25.3      27.4    -0.1
2806  2018-19  27.4      27.5      26.4      25.3      25.3    -2.1
3333  2019-20  25.3      27.4      27.5      26.4      25.0    -0.3
3868  2020-21  25.0      25.3      27.4      27.5      30.3     5.3
4449  2021-22  30.3      25.0      25.3      27.4      28.9    -1.4
5012  2022-23  28.9      30.3      25.0      25.3      25.7    -3.2
5583  2023-24  25.7      28.9      30.3      25.0      24.4    -1.3
6149  2024-25  24.4      25.7      28.9      30.3      20.9    -3.5
6724  2025-26  20.9      24.4      25.7      28.9       NaN     NaN
       SEASON   PTS  ppg_lag1  ppg_lag2  ppg_lag

In [8]:
df.loc[df["target"].idxmax()]

PLAYER_ID         1629663
SEASON            2021-22
PLAYER_NAME    Louis King
TEAM_ID        1610612758
TEAM_COUNT              1
AGE                  23.0
GP                     10
MIN                  10.4
PTS                   4.5
FGA                   4.7
FG_PCT              0.319
FG3A                  2.7
FG3_PCT             0.296
FTA                   1.0
FT_PCT                0.7
USG_PCT             0.195
TS_PCT              0.438
PACE                104.7
FG3A_total           27.0
FTA_total            10.0
season_year          2021
ppg_next             20.0
ppg_lag1              7.3
ppg_lag2              2.0
ppg_lag3              NaN
target               15.5
Name: 4456, dtype: object